In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/FINAL_PIPELINE_MODELS")

for p in ROOT.iterdir():
    print(p.name)

router_mnv3_router_v1_thr090.keras
rash_mnv3_stage2.keras
lesion_safety_bal_focal_best.keras
lesion_safety_config.json
gatekeeper_threshold.json
gatekeeper_float32.tflite
pipeline_cfg.json
acne5_severe_alert_threshold.json
acne5_severity_stage3.keras


IndentationError: unexpected indent (1005158164.py, line 252)

In [7]:
from google.colab import files

uploaded = files.upload()
test_image = list(uploaded.keys())[0]

res = run_skin_pipeline(test_image)
print_pipeline_result(res)


Saving images (1).jfif to images (1).jfif

================ PIPELINE RESULT ================
Image: images (1).jfif
Final decision: normal_or_non_skin
Guidance: Image appears normal or unsuitable for dermatology guidance.

--- Decision steps ---

model_1_gatekeeper:
  p_abnormal: 0.2081024944782257
  threshold: 0.22999999999999998
  decision: normal_or_non_skin


In [8]:
THRESHOLDS["gatekeeper_abnormal"] = 0.10

In [9]:
res = run_skin_pipeline(test_image)
print_pipeline_result(res)


================ PIPELINE RESULT ================
Image: images (1).jfif
Final decision: acne_moderate
Guidance: Detected acne. Severe-alert flag is positive; clinician review is recommended.

--- Decision steps ---

model_1_gatekeeper:
  p_abnormal: 0.2081024944782257
  threshold: 0.1
  decision: abnormal_skin

model_2_router:
  p_lesion: 0.47202184796333313
  threshold: 0.9
  decision: rash_branch

model_4_rash_specialist:
  class: acne
  class_index: 0
  confidence: 0.9998279809951782
  probabilities: {'acne': 0.9998279809951782, 'eczema': 0.000171430321643129, 'fungal': 6.312989171419758e-07}

model_5_acne_severity:
  class: moderate
  class_index: 2
  confidence: 0.31067153811454773
  probabilities: {'normal': 0.1209164708852768, 'mild': 0.2781984508037567, 'moderate': 0.31067153811454773, 'severe': 0.2902134954929352}
  p_severe: 0.2902134954929352
  severe_alert_threshold: 0.265
  severe_alert: True


In [15]:
from google.colab import files

uploaded = files.upload()
test_image = list(uploaded.keys())[0]

res = run_skin_pipeline(test_image)
print_pipeline_result(res)


Saving C0570529-Basal_cell_carcinoma_copy.original.max-600x600.jpg to C0570529-Basal_cell_carcinoma_copy.original.max-600x600.jpg

================ PIPELINE RESULT ================
Image: C0570529-Basal_cell_carcinoma_copy.original.max-600x600.jpg
Final decision: fungal
Guidance: Detected rash category: fungal.

--- Decision steps ---

model_1_gatekeeper:
  p_abnormal: 0.2081024944782257
  threshold: 0.1
  decision: abnormal_skin

model_2_router:
  p_lesion: 0.02040855400264263
  threshold: 0.9
  decision: rash_branch

model_4_rash_specialist:
  class: fungal
  class_index: 2
  confidence: 0.782920777797699
  probabilities: {'acne': 0.00020900437084492296, 'eczema': 0.21687020361423492, 'fungal': 0.782920777797699}


In [17]:
# ============================================================
# FINAL SKIN-APP INFERENCE CONTROLLER — COLAB VERSION UPDATED
# ============================================================

from pathlib import Path
import json
import numpy as np
import tensorflow as tf
from PIL import Image

# ------------------------------------------------------------
# 1. Final models folder
# ------------------------------------------------------------
FINAL_DIR = Path("/content/drive/MyDrive/FINAL_PIPELINE_MODELS")
assert FINAL_DIR.exists(), f"Folder not found: {FINAL_DIR}"

MODEL_PATHS = {
    "gatekeeper": FINAL_DIR / "gatekeeper_float32.tflite",
    "router": FINAL_DIR / "router_mnv3_router_v1_thr090.keras",
    "lesion_safety": FINAL_DIR / "lesion_safety_bal_focal_best.keras",
    "rash": FINAL_DIR / "rash_mnv3_stage2.keras",
    "severity": FINAL_DIR / "acne5_severity_stage3.keras",
}

CONFIG_PATHS = {
    "gatekeeper_threshold": FINAL_DIR / "gatekeeper_threshold.json",
    "lesion_safety_config": FINAL_DIR / "lesion_safety_config.json",
    "acne5_severe_alert_threshold": FINAL_DIR / "acne5_severe_alert_threshold.json",
    "pipeline_cfg": FINAL_DIR / "pipeline_cfg.json",
}

print("Files in final folder:")
for p in FINAL_DIR.iterdir():
    print(" -", p.name)

print("\nModel paths:")
for k, p in MODEL_PATHS.items():
    print(f"{k:15s}: {p.exists()} | {p.name}")

# ------------------------------------------------------------
# 2. Thresholds
# ------------------------------------------------------------
THRESHOLDS = {
    # Use 0.23 for real mode.
    # Use 0.10 only for branch-debugging.
    "gatekeeper_abnormal": 0.23,

    # IMPORTANT:
    # Router output behaves as p_rash / non-lesion.
    # Therefore:
    #   router_score < 0.90 => lesion branch
    #   router_score >= 0.90 => rash branch
    "router_rash_threshold": 0.90,

    "lesion_risk": 0.3438,
    "acne_severe_alert": 0.265,
}

RASH_CONF_THRESHOLD = 0.80

# Set this True to force abnormal and test downstream branches
BYPASS_GATEKEEPER = False

def safe_load_json(path):
    try:
        if path.exists():
            with open(path, "r") as f:
                return json.load(f)
    except Exception as e:
        print(f"Could not read {path.name}: {e}")
    return None

gate_json = safe_load_json(CONFIG_PATHS["gatekeeper_threshold"])
if gate_json:
    for key in ["threshold", "tau", "gatekeeper_abnormal"]:
        if key in gate_json:
            THRESHOLDS["gatekeeper_abnormal"] = float(gate_json[key])

lesion_json = safe_load_json(CONFIG_PATHS["lesion_safety_config"])
if lesion_json:
    for key in ["threshold", "tau", "lesion_risk"]:
        if key in lesion_json:
            THRESHOLDS["lesion_risk"] = float(lesion_json[key])

acne_json = safe_load_json(CONFIG_PATHS["acne5_severe_alert_threshold"])
if acne_json:
    for key in ["threshold", "tau", "severe_threshold", "p_severe_threshold"]:
        if key in acne_json:
            THRESHOLDS["acne_severe_alert"] = float(acne_json[key])

print("\nThresholds:")
for k, v in THRESHOLDS.items():
    print(f"{k:25s}: {v}")
print("rash_conf_threshold      :", RASH_CONF_THRESHOLD)
print("bypass_gatekeeper        :", BYPASS_GATEKEEPER)

# ------------------------------------------------------------
# 3. Class names
# ------------------------------------------------------------
CLASS_NAMES = {
    "rash": ["acne", "eczema", "fungal"],
    "severity_3": ["mild", "moderate", "severe"],
    "severity_4": ["normal", "mild", "moderate", "severe"],
}

IMG_SIZE = (224, 224)

# ------------------------------------------------------------
# 4. Load models
# ------------------------------------------------------------
def load_keras_model(path):
    if not path.exists():
        print(f"Missing model: {path}")
        return None
    return tf.keras.models.load_model(path, compile=False)

router_model = load_keras_model(MODEL_PATHS["router"])
lesion_model = load_keras_model(MODEL_PATHS["lesion_safety"])
rash_model = load_keras_model(MODEL_PATHS["rash"])
severity_model = load_keras_model(MODEL_PATHS["severity"])

gate_interpreter = None
gate_input = None
gate_output = None

if MODEL_PATHS["gatekeeper"].exists():
    gate_interpreter = tf.lite.Interpreter(model_path=str(MODEL_PATHS["gatekeeper"]))
    gate_interpreter.allocate_tensors()
    gate_input = gate_interpreter.get_input_details()[0]
    gate_output = gate_interpreter.get_output_details()[0]

print("\nLoaded models:")
print("Gatekeeper:", gate_interpreter is not None)
print("Router:", router_model is not None)
print("Lesion safety:", lesion_model is not None)
print("Rash specialist:", rash_model is not None)
print("Severity:", severity_model is not None)

# ------------------------------------------------------------
# 5. Image preprocessing
# ------------------------------------------------------------
def load_image_255(image_path):
    img = Image.open(image_path).convert("RGB")
    img = img.resize(IMG_SIZE)
    arr = np.array(img).astype("float32")
    arr = np.expand_dims(arr, axis=0)
    return arr

def sigmoid_output(pred):
    pred = np.array(pred).ravel()
    return float(pred[0])

def softmax_output(pred):
    return np.array(pred).ravel()

# ------------------------------------------------------------
# 6. Prediction helpers
# ------------------------------------------------------------
def predict_gatekeeper(image_255):
    if BYPASS_GATEKEEPER:
        return 1.0

    if gate_interpreter is None:
        return None

    # Gatekeeper final fix: [0,1]
    x = (image_255 / 255.0).astype(gate_input["dtype"])

    gate_interpreter.set_tensor(gate_input["index"], x)
    gate_interpreter.invoke()
    pred = gate_interpreter.get_tensor(gate_output["index"])

    return sigmoid_output(pred)

def predict_binary(model, image_255):
    pred = model.predict(image_255, verbose=0)
    return sigmoid_output(pred)

def predict_multiclass(model, image_255, class_names):
    probs = softmax_output(model.predict(image_255, verbose=0))
    idx = int(np.argmax(probs))

    return {
        "class": class_names[idx],
        "class_index": idx,
        "confidence": float(probs[idx]),
        "probabilities": {
            class_names[i]: float(probs[i]) for i in range(len(class_names))
        }
    }

# ------------------------------------------------------------
# 7. Main pipeline
# ------------------------------------------------------------
def run_skin_pipeline(image_path):
    image_path = Path(image_path)
    assert image_path.exists(), f"Image not found: {image_path}"

    image_255 = load_image_255(image_path)

    result = {
        "image_path": str(image_path),
        "thresholds": THRESHOLDS.copy(),
        "rash_conf_threshold": RASH_CONF_THRESHOLD,
        "bypass_gatekeeper": BYPASS_GATEKEEPER,
        "steps": {}
    }

    # -------------------------
    # Model-1 Gatekeeper
    # -------------------------
    p_abnormal = predict_gatekeeper(image_255)

    if p_abnormal is not None:
        is_abnormal = p_abnormal >= THRESHOLDS["gatekeeper_abnormal"]

        result["steps"]["model_1_gatekeeper"] = {
            "p_abnormal": p_abnormal,
            "threshold": THRESHOLDS["gatekeeper_abnormal"],
            "decision": "abnormal_skin" if is_abnormal else "normal_or_non_skin"
        }

        if not is_abnormal:
            result["final_decision"] = "normal_or_non_skin"
            result["guidance"] = "Image appears normal or unsuitable for dermatology guidance."
            return result
    else:
        result["steps"]["model_1_gatekeeper"] = {
            "warning": "Gatekeeper missing; skipped."
        }

    # -------------------------
    # Model-2 Router
    # -------------------------
    if router_model is None:
        result["final_decision"] = "error_router_missing"
        result["guidance"] = "Router model is missing."
        return result

    router_score = predict_binary(router_model, image_255)

    # IMPORTANT FIX:
    # Router score behaves as p_rash/non-lesion.
    # Low = lesion, high = rash.
    is_lesion = router_score < THRESHOLDS["router_rash_threshold"]

    result["steps"]["model_2_router"] = {
        "router_score": router_score,
        "interpreted_as": "p_rash_or_non_lesion",
        "threshold": THRESHOLDS["router_rash_threshold"],
        "decision": "lesion_branch" if is_lesion else "rash_branch"
    }

    # -------------------------
    # Lesion branch — Model-3
    # -------------------------
    if is_lesion:
        if lesion_model is None:
            result["final_decision"] = "error_lesion_model_missing"
            result["guidance"] = "Lesion branch selected but lesion safety model is missing."
            return result

        p_risk = predict_binary(lesion_model, image_255)
        is_risk = p_risk >= THRESHOLDS["lesion_risk"]

        result["steps"]["model_3_lesion_safety"] = {
            "p_risk": p_risk,
            "threshold": THRESHOLDS["lesion_risk"],
            "decision": "risk" if is_risk else "safe_or_lower_risk"
        }

        if is_risk:
            result["final_decision"] = "suspicious_lesion_risk"
            result["guidance"] = "Safety alert: this lesion should be reviewed by a clinician."
        else:
            result["final_decision"] = "lower_risk_lesion"
            result["guidance"] = "The lesion appears lower risk, but clinical review is advised if it changes, bleeds, hurts, or looks irregular."

        return result

    # -------------------------
    # Rash branch — Model-4
    # -------------------------
    if rash_model is None:
        result["final_decision"] = "error_rash_model_missing"
        result["guidance"] = "Rash branch selected but rash specialist model is missing."
        return result

    rash_pred = predict_multiclass(rash_model, image_255, CLASS_NAMES["rash"])
    result["steps"]["model_4_rash_specialist"] = rash_pred

    # Low confidence rejection to reduce normal-face false rash outputs
    if rash_pred["confidence"] < RASH_CONF_THRESHOLD:
        result["final_decision"] = "uncertain_or_normal_skin"
        result["guidance"] = "No confident rash pattern detected. Please retake the image or consult a clinician if symptoms are present."
        return result

    rash_class = rash_pred["class"]

    # -------------------------
    # Acne branch — Model-5
    # -------------------------
    if rash_class == "acne":
        if severity_model is None:
            result["final_decision"] = "acne_detected_severity_missing"
            result["guidance"] = "Acne detected but severity model is missing."
            return result

        severity_probs = softmax_output(severity_model.predict(image_255, verbose=0))

        if len(severity_probs) == 3:
            sev_names = CLASS_NAMES["severity_3"]
        elif len(severity_probs) == 4:
            sev_names = CLASS_NAMES["severity_4"]
        else:
            sev_names = [f"class_{i}" for i in range(len(severity_probs))]

        sev_idx = int(np.argmax(severity_probs))
        sev_class = sev_names[sev_idx]

        severity_result = {
            "class": sev_class,
            "class_index": sev_idx,
            "confidence": float(severity_probs[sev_idx]),
            "probabilities": {
                sev_names[i]: float(severity_probs[i]) for i in range(len(sev_names))
            }
        }

        if "severe" in sev_names:
            severe_idx = sev_names.index("severe")
            p_severe = float(severity_probs[severe_idx])
            severe_alert = p_severe >= THRESHOLDS["acne_severe_alert"]

            severity_result["p_severe"] = p_severe
            severity_result["severe_alert_threshold"] = THRESHOLDS["acne_severe_alert"]
            severity_result["severe_alert"] = severe_alert

        result["steps"]["model_5_acne_severity"] = severity_result
        result["final_decision"] = f"acne_{sev_class}"

        if severity_result.get("severe_alert", False):
            result["guidance"] = "Detected acne. Severe-alert flag is positive; clinician review is recommended."
        else:
            result["guidance"] = f"Detected acne with predicted severity: {sev_class}."

        return result

    # -------------------------
    # Non-acne rash
    # -------------------------
    result["final_decision"] = rash_class
    result["guidance"] = f"Detected rash category: {rash_class}."

    return result

# ------------------------------------------------------------
# 8. Pretty printer
# ------------------------------------------------------------
def print_pipeline_result(result):
    print("\n================ PIPELINE RESULT ================")
    print("Image:", result.get("image_path"))
    print("Final decision:", result.get("final_decision"))
    print("Guidance:", result.get("guidance"))

    print("\n--- Decision steps ---")
    for step, info in result.get("steps", {}).items():
        print(f"\n{step}:")
        for k, v in info.items():
            print(f"  {k}: {v}")

# ------------------------------------------------------------
# 9. Upload and test helper
# ------------------------------------------------------------
def upload_and_test():
    from google.colab import files
    uploaded = files.upload()
    image_path = list(uploaded.keys())[0]
    res = run_skin_pipeline(image_path)
    print_pipeline_result(res)
    return res

# Usage:
# res = upload_and_test()
# or:
# test_image = "/content/your_image.jpg"
# res = run_skin_pipeline(test_image)
# print_pipeline_result(res)

Files in final folder:
 - router_mnv3_router_v1_thr090.keras
 - rash_mnv3_stage2.keras
 - lesion_safety_bal_focal_best.keras
 - lesion_safety_config.json
 - gatekeeper_threshold.json
 - gatekeeper_float32.tflite
 - pipeline_cfg.json
 - acne5_severe_alert_threshold.json
 - acne5_severity_stage3.keras

Model paths:
gatekeeper     : True | gatekeeper_float32.tflite
router         : True | router_mnv3_router_v1_thr090.keras
lesion_safety  : True | lesion_safety_bal_focal_best.keras
rash           : True | rash_mnv3_stage2.keras
severity       : True | acne5_severity_stage3.keras

Thresholds:
gatekeeper_abnormal      : 0.22999999999999998
router_rash_threshold    : 0.9
lesion_risk              : 0.34375691413879395
acne_severe_alert        : 0.265
rash_conf_threshold      : 0.8
bypass_gatekeeper        : False

Loaded models:
Gatekeeper: True
Router: True
Lesion safety: True
Rash specialist: True
Severity: True


In [18]:
res = upload_and_test()

Saving C0570529-Basal_cell_carcinoma_copy.original.max-600x600.jpg to C0570529-Basal_cell_carcinoma_copy.original.max-600x600 (1).jpg

================ PIPELINE RESULT ================
Image: C0570529-Basal_cell_carcinoma_copy.original.max-600x600 (1).jpg
Final decision: normal_or_non_skin
Guidance: Image appears normal or unsuitable for dermatology guidance.

--- Decision steps ---

model_1_gatekeeper:
  p_abnormal: 0.2081024944782257
  threshold: 0.22999999999999998
  decision: normal_or_non_skin
